# 00 — Preparazione dello split comune (dataset RF reale)

Questo notebook va eseguito **una volta**. Produce i file `splits/split_group_v1.npz` e `splits/split_block_v1.npz`, che **tutti** i notebook dei modelli caricano con `qdetect.load_split`: così ogni modello (RF, VQE, QAOA, VQS, stump, QUBO, ...) vede esattamente le stesse righe di train, validation e test, con lo stesso preprocessing.

Passi:
1. campione uniforme per classe dal CSV, letto a blocchi (senza caricare l'intero file);
2. diagnostica: duplicati, conflitti di etichetta (limite inferiore all'errore), ordine del file;
3. split `group` (nessun vettore di feature identico tra insiemi) e split `block` (blocchi contigui del file + purge, controllo di robustezza);
4. controllo con una Random Forest: se l'AUROC crolla passando da `group` a `block`, righe vicine nel file sono correlate e lo split `group` è ottimista.

In [ ]:
# Colab / server: rendere importabile il pacchetto qdetect della repo.
# Su Colab:  !git clone https://github.com/lnrdmnc/Quantum-Enhanced-Detection-Mechanisms-for-Mobile-Network-Security.git repo
#            %cd repo
#            %pip install -q -e .
import sys
from pathlib import Path
REPO = Path.cwd()
while not (REPO / 'qdetect').exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

import json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import qdetect as qd
print('repo:', REPO)

In [ ]:
DATA_PATH = Path('output/split/train_dataset.csv')   # CSV da ~2 GB, 52,7 M righe
OUT_DIR = REPO / 'splits'

CAP_PER_CLASS = 120_000   # righe per classe nel campione (come nel notebook QAOA v2)
SAMPLE_SEED = 2026
SPLIT_SEED = 42
SIZES = dict(n_train=5_000, n_train_large=20_000, n_val=1_000, n_test=5_000)

assert DATA_PATH.exists(), f'Dataset non trovato: {DATA_PATH.resolve()}'
print(DATA_PATH.resolve(), round(DATA_PATH.stat().st_size / 1e9, 2), 'GB')

## 1. Campionamento

In [ ]:
t0 = time.time()
df, sampling = qd.sample_per_class(DATA_PATH, CAP_PER_CLASS, seed=SAMPLE_SEED)
print(f'{time.time()-t0:.0f} s')
print(json.dumps(sampling, indent=1))
print('campione:', df.shape)
df.head()

## 2. Diagnostica

`bayes_error_floor`: frazione di errori che **nessun** classificatore basato su queste 8 feature può evitare, perché lo stesso vettore compare con entrambe le etichette. Serve a leggere correttamente accuratezze del 95–96%.

In [ ]:
rep = qd.duplicate_report(df[qd.FEATURES].to_numpy(float), df[qd.TARGET].to_numpy())
print(json.dumps(rep, indent=1))

In [ ]:
# Distribuzione delle classi per frequenza e statistiche per classe
display(pd.crosstab(df['freq1'], df[qd.TARGET], normalize='index').round(3))
display(df.groupby(qd.TARGET)[qd.FEATURES].describe().T.round(2))

In [ ]:
# Il file è in ordine di acquisizione? Se la proporzione di malevoli varia a blocchi
# lungo il file, le righe vicine sono correlate e lo split 'block' è il controllo giusto.
s = df.sort_values('row_id')
w = 2000
plt.figure(figsize=(9, 3))
plt.plot(s['row_id'].to_numpy()[w-1:], s[qd.TARGET].rolling(w).mean().to_numpy()[w-1:])
plt.xlabel('posizione nel CSV (row_id)'); plt.ylabel(f'frazione malevoli (media mobile {w})')
plt.tight_layout(); plt.show()

## 3. Split `group` e `block`

In [ ]:
splits = {}
for strategy in ['group', 'block']:
    sp = qd.make_split(df, strategy=strategy, seed=SPLIT_SEED, **SIZES)
    path = qd.save_split(sp, OUT_DIR / f'split_{strategy}_v1.npz',
                         extra_meta={'data_path': str(DATA_PATH), 'sampling': sampling,
                                     'duplicates': rep, 'sizes': SIZES})
    splits[strategy] = sp
    print(strategy, '→', path, '| pool:', sp.meta['pool_sizes'])
    display(sp.summary())

## 4. Controllo con Random Forest

Stessa RF su entrambi gli split. Non è un risultato della tesi, solo un controllo di sanità dello split.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rows = []
for strategy, sp in splits.items():
    P = qd.prepare(sp, fit_on='train_large')
    rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=2, n_jobs=-1, random_state=0)
    rf.fit(P.X_train_large, P.y_train_large)
    rows.append(qd.evaluate('RF check', P.y_test, rf.predict_proba(P.X_test)[:, 1],
                            family='classical', split=strategy, n_train=len(P.y_train_large)))
pd.DataFrame(rows)[['split', 'auroc', 'auroc_lo', 'auroc_hi', 'f1', 'pd_at_pfa_0.01']].round(4)

## Uso nei notebook dei modelli

```python
import qdetect as qd
split = qd.load_split(REPO / 'splits/split_group_v1.npz')
P = qd.prepare(split, angle_scale=2.0)            # 8 feature → angoli per l'embedding
# P = qd.prepare(split, k_best=4, angle_scale=2.0) # variante a 4 qubit
...
row = qd.evaluate('QAOA 8q', P.y_test, scores, family='quantum', seed=seed,
                  n_train=len(P.y_train), n_params=n_params,
                  train_seconds=t_train, infer_seconds=t_inf)
qd.append_results(REPO / 'results/all_results.csv', [row])
```

Convenzione: il punteggio passato a `evaluate` è **alto = malevolo**. Per i modelli a energia, il segno si fissa con `qd.orient_scores` su train o validation, mai sul test.